# Desafio 2
## Gurmendi Alan

## Consigna:

* Crear sus propios vectores con Gensim basado en lo visto en clase con otro artista del dataset Songs
* Elegir terminos de interes y buscar terminos mas similares y menos similares
* Realizar una reduccion de dimensionalidad de embeddings, llevandolos a 2 dimensiones. Graficar los embeddings proyectados y seleccionar una cantidad de terminos (variable MAX_WORDS) de forma tal que la visualizacion sea adecuada.
* Inpeccionar el grafico y buscar pequeños grupos de palabras que puedan formarse. Interpretarlos e intentar obtener conclusiones. En lo posible, acompañar los grupos de palabras con capturas (y pegarlas en celdas de texto)

---

Se Realizan los imports necesarios

In [1]:
from gensim.models import Word2Vec
from tensorflow.keras.preprocessing.text import text_to_word_sequence
from gensim.models.callbacks import CallbackAny2Vec
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import multiprocessing


2025-11-13 09:10:37.833783: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-13 09:10:37.837745: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-13 09:10:38.024836: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-13 09:10:39.146776: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation or

Se descarga el dataset provisto por la catedra 

In [2]:
import os
import platform
if os.access('./songs_dataset', os.F_OK) is False:
    if os.access('songs_dataset.zip', os.F_OK) is False:
        if platform.system() == 'Windows':
            !curl https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip -o songs_dataset.zip
        else:
            !wget songs_dataset.zip https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
    !unzip -q songs_dataset.zip   
else:
    print("El dataset ya se encuentra descargado")

El dataset ya se encuentra descargado


Para realizar este desafío se van a utilizar las letras de las canciones de Bob Marley. No soy un conocedor profundo de sus canciones, pero sí de las temáticas que las caracterizan. Por eso, el análisis de la cercanía entre palabras se realizará desde una perspectiva semántica y temática, priorizando el sentido general de los conceptos por encima de su aparición en canciones específicas.

In [3]:
# Armar el dataset utilizando salto de línea para separar las oraciones/docs
df = pd.read_csv('songs_dataset/bob-marley.txt', sep='/n', header=None)
df.head()

/tmp/ipykernel_7905/920653701.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv('songs_dataset/bob-marley.txt', sep='/n', header=None)


,0
0,"""Don't worry about a thing,"
1,'Cause every little thing gonna be all right.
2,"Singin': ""Don't worry about a thing,"
3,"'Cause every little thing gonna be all right!""..."
4,"Smiled with the risin' sun,"


In [4]:
print("Cantidad de documentos:", df.shape[0])

Cantidad de documentos: 2218


Se tokenizan los documentos

In [5]:
sentence_tokens = []
for _, row in df[:None].iterrows():
    sentence_tokens.append(text_to_word_sequence(row[0]))

sentence_tokens[:2]

[["don't", 'worry', 'about', 'a', 'thing'],
 ["'cause", 'every', 'little', 'thing', 'gonna', 'be', 'all', 'right']]

In [6]:
class callback(CallbackAny2Vec):
    """
    Callback to print loss after each epoch
    """
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss after epoch {}: {}'.format(self.epoch, loss))
        else:
            print('Loss after epoch {}: {}'.format(self.epoch, loss- self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

Se crea el modelo

In [7]:
w2v_model = Word2Vec(min_count=5,    # frecuencia mínima de palabra para incluirla en el vocabulario
                     window=2,       # cant de palabras antes y desp de la predicha
                     vector_size=300,       # dimensionalidad de los vectores 
                     negative=20,    # cantidad de negative samples... 0 es no se usa
                     workers=1,      # si tienen más cores pueden cambiar este valor
                     sg=1)           # modelo 0:CBOW  1:skipgram

In [8]:
# Obtener el vocabulario con los tokens
w2v_model.build_vocab(sentence_tokens)

In [9]:
# Cantidad de filas/docs encontradas en el corpus
print("Cantidad de docs en el corpus:", w2v_model.corpus_count)

Cantidad de docs en el corpus: 2218


In [10]:
# Cantidad de words encontradas en el corpus
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))

Cantidad de words distintas en el corpus: 532


Se entrena el modelo

In [11]:
w2v_model.train(sentence_tokens,
                 total_examples=w2v_model.corpus_count,
                 epochs=20,
                 compute_loss = True,
                 callbacks=[callback()]
                 )

Loss after epoch 0: 150191.328125
Loss after epoch 1: 93119.09375
Loss after epoch 2: 92445.671875
Loss after epoch 3: 92579.71875
Loss after epoch 4: 92034.375
Loss after epoch 5: 88941.375
Loss after epoch 6: 82685.375
Loss after epoch 7: 79544.25
Loss after epoch 8: 75712.625
Loss after epoch 9: 72144.25
Loss after epoch 10: 70732.0
Loss after epoch 11: 68090.4375
Loss after epoch 12: 62246.25
Loss after epoch 13: 60418.625
Loss after epoch 14: 60237.125
Loss after epoch 15: 59959.625
Loss after epoch 16: 58845.5
Loss after epoch 17: 58793.625
Loss after epoch 18: 58291.5
Loss after epoch 19: 57499.875


(218093, 359260)

In [12]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["love"], topn=10)

[('protect', 0.8086313009262085),
 ('nobody', 0.8049787878990173),
 ('given', 0.8032711148262024),
 ("you've", 0.799490213394165),
 ('really', 0.7939767241477966),
 ("feelin'", 0.7812795042991638),
 ('message', 0.7800683975219727),
 ('ask', 0.7798995971679688),
 ('mean', 0.7631951570510864),
 ('door', 0.745693027973175)]

In [13]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["jamaica"], topn=10)

[("y'all", 0.963806688785553),
 ('smile', 0.933390736579895),
 ('wise', 0.877193808555603),
 ('yard', 0.8121813535690308),
 ('together', 0.7862884998321533),
 ('get', 0.7715235948562622),
 ("let's", 0.7703178524971008),
 ('top', 0.7700229287147522),
 ("they're", 0.763706624507904),
 ('name', 0.7630559206008911)]

In [14]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["cry"], topn=10)

[('woman', 0.975132405757904),
 ('no', 0.9103832244873047),
 ('fowl', 0.9090774059295654),
 ('tears', 0.8873773217201233),
 ('matter', 0.8803579211235046),
 ('call', 0.875583827495575),
 ("ain't", 0.8359548449516296),
 ('hypocrites', 0.8180119395256042),
 ('shed', 0.8141749501228333),
 ('nice', 0.8140014410018921)]

In [15]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["life"], topn=10)

[('road', 0.8402378559112549),
 ("feelin'", 0.7735351324081421),
 ('war', 0.7625907063484192),
 ('fire', 0.7619547247886658),
 ('really', 0.7592171430587769),
 ('is', 0.7580807209014893),
 ('else', 0.7577210664749146),
 ('this', 0.756687581539154),
 ('brought', 0.755793035030365),
 ('judging', 0.7481594085693359)]

In [16]:
# el método `get_vector` permite obtener los vectores:
vector_life = w2v_model.wv.get_vector("life")
print(vector_life)

[ 0.22997366 -0.04581723 -0.20601924 -0.21116075 -0.25791594 -0.21739411
  0.32134232  0.13975343 -0.05148359  0.04180424 -0.0215387   0.06379415
 -0.22212364  0.12927116 -0.13661139 -0.09325753  0.05482211  0.10144574
 -0.15923575 -0.24094151 -0.21512382 -0.00822827  0.20031546  0.34429726
 -0.27415588 -0.10923047 -0.13419591 -0.19695191  0.18053058 -0.26333678
  0.01085525  0.07238078  0.0855068   0.20380688  0.40724537  0.02010075
  0.15961087 -0.03145773  0.22512023  0.26752776 -0.18422484 -0.31916773
  0.2907701  -0.23206234  0.05246921  0.07464758  0.03419438 -0.07603288
  0.3013976  -0.18721536 -0.22617452  0.1552324   0.33702096 -0.02710209
 -0.2520683  -0.12623125 -0.07041165  0.12177835  0.05861206 -0.14581643
 -0.19228286 -0.2535293  -0.03058918  0.210663    0.12395918 -0.122712
 -0.17748214  0.18854985  0.26560932  0.00375436 -0.24116461 -0.24569584
  0.16155985 -0.06211389  0.08014258 -0.0265058   0.00987349  0.2228252
  0.13739054  0.40071788 -0.13626252 -0.09580113 -0.03

In [17]:
# el método `most_similar` también permite comparar a partir de vectores
w2v_model.wv.most_similar(vector_life)

[('life', 1.0000001192092896),
 ('road', 0.8402378559112549),
 ("feelin'", 0.7735351324081421),
 ('war', 0.7625907063484192),
 ('fire', 0.7619547247886658),
 ('really', 0.7592171430587769),
 ('is', 0.7580807209014893),
 ('else', 0.7577210068702698),
 ('this', 0.756687581539154),
 ('brought', 0.755793035030365)]

Se realiza la reducción de dimensionalidad a dos dimensiones

In [18]:
from sklearn.decomposition import IncrementalPCA    
from sklearn.manifold import TSNE                   
import numpy as np                                  

def reduce_dimensions(model, num_dimensions = 2 ):
     
    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)  

    tsne = TSNE(n_components=num_dimensions, random_state=0)
    vectors = tsne.fit_transform(vectors)

    return vectors, labels

Se realiza el grafico en 2D. Para este caso se elije una cantidad de 400 palabras para poder visualizar casi en completo el cojunto de palabras

In [19]:
# Graficar los embedddings en 2D
import plotly.graph_objects as go
import plotly.express as px

vecs, labels = reduce_dimensions(w2v_model)

MAX_WORDS=400
fig = px.scatter(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], text=labels[:MAX_WORDS])
fig.show(renderer="colab") # esto para plotly en colab

### Grupos de palabras encontrados:

**america, war, work, morning**

![Embeddings](imagenes_desafio_2/grupo_1.png)

En este caso se observa la asociación de “america” con “war” y “work”, lo que refleja la presencia de una temática social y de conflicto en las letras.

---

**move, movement, people, exodus, jah**

![Embeddings](imagenes_desafio_2/grupo_2.png)

En este grupo las palabras “move”, “movement”, “people”, “exodus” y “jah” reflejan una temática de libertad, unión y espiritualidad, muy presente en las letras de Bob Marley y la cultura Reggae

---

**woman, cry, no, tears**

![Embeddings](imagenes_desafio_2/grupo_3.png)

En este grupo las palabras hacen referencia directa a la canción “No Woman, No Cry”

---

**vibration, positive, rastaman**

![Embeddings](imagenes_desafio_2/grupo_4.png)

En este caso las palabras hacen referencia a la cultura reggae y al movimiento rastafari



